# Healthcare Analytics for Doctor Visits
### Exploratory Data Analysis, Statistical Testing, and Predictive Modelling

**Project owner:** Mohammad Qutbuddin  
**Objective:** Analyse patient-level data to identify factors associated with the number of doctor visits and build transparent baseline models that estimate visit frequency.

---

## Notebook operating principles

- **Evidence-first:** the presentation's reported figures are treated as hypotheses to validate against the raw dataset, not as hard-coded results.
- **Reproducibility:** fixed random seeds, explicit configuration, reusable helper functions, and saved outputs.
- **Healthcare caution:** this is an observational analysis. Associations do not establish causation, and predictions are not clinical advice.
- **Data protection:** do not use direct patient identifiers in modelling. This notebook reports aggregate patterns only.

**Expected data:** a CSV containing approximately 5,190 records and variables such as `visits`, `gender`, `age`, `income`, `illness`, `reduced`, `health`, `private`, `freepoor`, `freerepat`, `nchronic`, and `lchronic`. Column names may differ; the profiling cells help identify mismatches.

## 1. Environment and reproducibility
Install dependencies if needed. In an existing Jupyter environment, uncomment the installation command only if packages are missing.

In [ ]:
# If required, run in a separate notebook cell:
# %pip install pandas numpy matplotlib seaborn scipy scikit-learn statsmodels joblib

In [ ]:
from pathlib import Path
import json
import warnings
import sys
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix
)
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
import joblib

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 2. Project configuration and data loading

Place the dataset CSV in the same folder as this notebook, or update `DATA_PATH`. The loader searches common filenames if the default path is absent. It intentionally does not silently download data from an unverified source.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "DoctorVisits.csv"  # Change this to your actual CSV path
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    candidates = [
        PROJECT_DIR / "doctorvisits.csv",
        PROJECT_DIR / "Doctor_Visits.csv",
        PROJECT_DIR / "doctor_visits.csv",
        PROJECT_DIR / "DoctorVisitsData.csv",
    ]
    found = next((p for p in candidates if p.exists()), None)
    if found:
        DATA_PATH = found
    else:
        csv_files = sorted(PROJECT_DIR.glob("*.csv"))
        if len(csv_files) == 1:
            DATA_PATH = csv_files[0]
        elif len(csv_files) > 1:
            print("CSV files found:")
            for p in csv_files:
                print(" -", p.name)
            raise FileNotFoundError(
                f"Set DATA_PATH explicitly. Multiple CSV files were found in {PROJECT_DIR}."
            )
        else:
            raise FileNotFoundError(
                f"No CSV found in {PROJECT_DIR}. Put your dataset here or update DATA_PATH."
            )

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")
display(df_raw.head())

## 3. Data dictionary and schema audit

The analysis expects a visit-count target and candidate demographic, health, income, and coverage predictors. The actual dataset is authoritative: review the discovered columns before proceeding.

In [ ]:
print("Column names:")
display(pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [str(x) for x in df_raw.dtypes],
    "missing_n": df_raw.isna().sum().values,
    "missing_pct": (df_raw.isna().mean().values * 100).round(2),
    "unique_n": df_raw.nunique(dropna=True).values,
}))

print("\nDescriptive statistics (all columns):")
display(df_raw.describe(include="all").T)

In [ ]:
# Standardize labels for reliable downstream references.
df = df_raw.copy()
df.columns = (
    df.columns.astype(str).str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Common aliases seen in versions of the doctor-visits dataset.
ALIASES = {
    "doctor_visits": "visits",
    "num_visits": "visits",
    "number_of_visits": "visits",
    "doctorvisits": "visits",
    "sex": "gender",
    "age_years": "age",
    "reduced_activity_days": "reduced",
    "days_reduced": "reduced",
    "health_status": "health",
    "private_insurance": "private",
    "free_poor": "freepoor",
    "free_repatriation": "freerepat",
    "chronic": "nchronic",
    "long_chronic": "lchronic",
}
df = df.rename(columns={c: ALIASES[c] for c in df.columns if c in ALIASES})

print("Normalized columns:", df.columns.tolist())
TARGET = "visits"
if TARGET not in df.columns:
    raise ValueError(
        "Could not identify the target column 'visits'. Rename your doctor-visit count column "
        "to 'visits' or add its alias to ALIASES, then rerun this cell."
    )

## 4. Data quality assessment and cleaning

Cleaning decisions are deliberately conservative. Duplicate rows and impossible target values are flagged first. Numeric conversion is attempted only for columns that should be numeric. Review any exclusions before using results in a report.

In [ ]:
quality = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_cells": int(df.isna().sum().sum()),
    "missing_pct_cells": round(float(df.isna().mean().mean() * 100), 2),
    "target_missing": int(df[TARGET].isna().sum()),
}
display(pd.Series(quality, name="value").to_frame())

print("Target distribution before cleaning:")
display(df[TARGET].value_counts(dropna=False).sort_index().to_frame("count"))

In [ ]:
# Drop exact duplicate records; document the number removed.
n_before = len(df)
df = df.drop_duplicates().copy()
print(f"Exact duplicate rows removed: {n_before - len(df)}")

# Convert likely numeric variables without forcing categorical variables into numeric codes.
numeric_candidates = [
    "visits", "age", "income", "illness", "reduced", "health",
    "nchronic", "lchronic", "actdays"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# A visit count should be non-negative. Invalid targets cannot be used for supervised modelling.
invalid_target = df[TARGET].notna() & (df[TARGET] < 0)
print("Rows with negative target values:", int(invalid_target.sum()))
df = df.loc[~invalid_target].copy()
df = df.loc[df[TARGET].notna()].copy()

# Keep a clean working copy; missing predictor values will be imputed inside model pipelines.
print("Analysis shape:", df.shape)
display(df.head())

## 5. Exploratory data analysis (EDA)

### 5.1 Target distribution and zero-visit prevalence
The presentation reports that approximately 80% of participants had zero visits. This section checks the figure using the loaded data.

In [ ]:
visit_counts = df[TARGET].value_counts().sort_index()
zero_share = float((df[TARGET] == 0).mean())
print(f"Zero-visit share: {zero_share:.1%}")
print(f"Mean visits: {df[TARGET].mean():.3f}")
print(f"Median visits: {df[TARGET].median():.3f}")
print(f"Maximum observed visits: {df[TARGET].max():.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x=TARGET, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of doctor visits")
axes[0].set_xlabel("Number of visits")
axes[0].set_ylabel("Number of people")

axes[1].bar(["Zero visits", "One or more"], [(df[TARGET] == 0).sum(), (df[TARGET] > 0).sum()],
            color=["slategray", "teal"])
axes[1].set_title("Zero versus non-zero visits")
axes[1].set_ylabel("Number of people")
for ax in axes:
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "target_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

### 5.2 Univariate distributions and missingness

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing.rename("missing_pct").to_frame())

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    df[numeric_cols].hist(figsize=(15, 10), bins=25, edgecolor="white")
    plt.suptitle("Numeric feature distributions", y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "numeric_distributions.png", dpi=160, bbox_inches="tight")
    plt.show()

### 5.3 Visits by demographic, health, and coverage variables
Means are easy to interpret but can be influenced by a small number of high-visit observations. Always inspect group sizes alongside averages.

In [ ]:
def grouped_visit_summary(data, feature, target=TARGET, max_categories=20):
    if feature not in data.columns:
        print(f"Skipping {feature}: column not present.")
        return None
    temp = data[[feature, target]].dropna().copy()
    if temp[feature].nunique() > max_categories:
        if pd.api.types.is_numeric_dtype(temp[feature]):
            try:
                temp[feature] = pd.qcut(temp[feature], q=4, duplicates="drop")
            except ValueError:
                pass
        else:
            print(f"Skipping {feature}: too many categories.")
            return None
    result = temp.groupby(feature, observed=False)[target].agg(
        people="size", mean_visits="mean", median_visits="median", visit_sd="std"
    ).reset_index()
    return result

candidate_features = [
    "gender", "age", "income", "illness", "reduced", "health",
    "nchronic", "lchronic", "private", "freepoor", "freerepat"
]
for feature in candidate_features:
    if feature in df.columns:
        print(f"\n{feature.upper()} vs visits")
        summary = grouped_visit_summary(df, feature)
        if summary is not None:
            display(summary.head(30))

In [ ]:
# Plot categorical features directly; bin continuous variables for readable comparisons.
for feature in ["gender", "private", "freepoor", "freerepat", "nchronic", "lchronic"]:
    if feature not in df.columns or df[feature].nunique(dropna=True) > 12:
        continue
    plot_data = df[[feature, TARGET]].dropna().copy()
    plt.figure(figsize=(7, 4))
    sns.barplot(data=plot_data, x=feature, y=TARGET, errorbar=("ci", 95), color="teal")
    plt.title(f"Mean doctor visits by {feature}")
    plt.xlabel(feature)
    plt.ylabel("Mean visits (95% CI)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"mean_visits_by_{feature}.png", dpi=160, bbox_inches="tight")
    plt.show()

for feature in ["age", "income", "illness", "reduced", "health"]:
    if feature not in df.columns or not pd.api.types.is_numeric_dtype(df[feature]):
        continue
    plot_data = df[[feature, TARGET]].dropna().copy()
    if plot_data[feature].nunique() > 10:
        try:
            plot_data["feature_bin"] = pd.qcut(plot_data[feature], q=5, duplicates="drop")
        except ValueError:
            continue
        xcol = "feature_bin"
    else:
        xcol = feature
    summary = plot_data.groupby(xcol, observed=False)[TARGET].mean().reset_index()
    plt.figure(figsize=(9, 4))
    sns.barplot(data=summary, x=xcol, y=TARGET, color="cornflowerblue")
    plt.title(f"Mean doctor visits by {feature}")
    plt.xlabel(feature)
    plt.ylabel("Mean visits")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"mean_visits_by_{feature}_binned.png", dpi=160, bbox_inches="tight")
    plt.show()

## 6. Association analysis

### 6.1 Correlations among numeric variables
Pearson correlation measures linear association and can miss non-linear relationships. Correlations with the visit count are descriptive, not causal evidence.

In [ ]:
corr_cols = df.select_dtypes(include=np.number).columns.tolist()
if len(corr_cols) >= 2:
    corr = df[corr_cols].corr(method="pearson")
    display(corr.round(3))
    plt.figure(figsize=(11, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=False)
    plt.title("Pearson correlation matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "correlation_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()

    if TARGET in corr.columns:
        target_corr = corr[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
        display(target_corr.rename("pearson_corr_with_visits").to_frame())

### 6.2 Simple inferential checks
These tests are exploratory. Welch's t-test compares two group means; Spearman correlation assesses monotonic association. P-values should be interpreted alongside effect sizes, sample sizes, multiple testing, and data limitations.

In [ ]:
# Welch t-test for a two-level gender variable, if applicable.
if "gender" in df.columns and df["gender"].nunique(dropna=True) == 2:
    levels = df["gender"].dropna().unique().tolist()
    group_a = df.loc[df["gender"] == levels[0], TARGET].dropna()
    group_b = df.loc[df["gender"] == levels[1], TARGET].dropna()
    test = stats.ttest_ind(group_a, group_b, equal_var=False, nan_policy="omit")
    print(f"Welch t-test: {levels[0]} (n={len(group_a)}) vs {levels[1]} (n={len(group_b)})")
    print(f"Mean difference ({levels[0]} - {levels[1]}): {group_a.mean() - group_b.mean():.4f}")
    print(f"t-statistic={test.statistic:.4f}, p-value={test.pvalue:.6g}")
else:
    print("Gender comparison skipped: a two-level 'gender' column was not found.")

# Spearman correlations for numeric features.
spearman_rows = []
for col in df.select_dtypes(include=np.number).columns:
    if col == TARGET:
        continue
    pair = df[[col, TARGET]].dropna()
    if len(pair) >= 3 and pair[col].nunique() > 1 and pair[TARGET].nunique() > 1:
        r, p = stats.spearmanr(pair[col], pair[TARGET])
        spearman_rows.append({"feature": col, "n": len(pair), "spearman_rho": r, "p_value": p})
spearman_results = pd.DataFrame(spearman_rows)
if not spearman_results.empty:
    display(spearman_results.sort_values("spearman_rho", key=lambda s: s.abs(), ascending=False))

## 7. Predictive modelling

### Modelling strategy
The target is a non-negative count and may contain many zeros. We compare a naive mean baseline with a Random Forest regressor and HistGradientBoosting regressor using a held-out test set. These are pragmatic predictive baselines, not a claim that a generic regressor is the ideal statistical model for count data.

- Split data before fitting preprocessing to avoid leakage.
- Median-impute numeric values and most-frequent-impute categorical values.
- One-hot encode categorical predictors.
- Exclude the target and obvious identifier-like columns.
- Evaluate MAE and RMSE; \(R^2\) is reported but may be negative on a held-out set.

In [ ]:
# Exclude target and columns that look like record identifiers.
id_like = [c for c in df.columns if c in {"id", "patient_id", "person_id", "record_id"} or c.endswith("_id")]
feature_cols = [c for c in df.columns if c != TARGET and c not in id_like]
model_df = df[feature_cols + [TARGET]].copy()

# Avoid leakage from rows with missing target; predictors can be imputed in pipelines.
model_df = model_df.dropna(subset=[TARGET])
X = model_df[feature_cols]
y = model_df[TARGET].astype(float)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
], remainder="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric predictors:", numeric_features)
print("Categorical predictors:", categorical_features)

In [ ]:
models = {
    "Dummy mean baseline": DummyRegressor(strategy="mean"),
    "Random Forest": RandomForestRegressor(
        n_estimators=400, min_samples_leaf=3, random_state=SEED,
        n_jobs=-1, max_features=1.0
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.06, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=SEED
    )
}

fitted_models = {}
metric_rows = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = np.clip(pipe.predict(X_test), 0, None)
    fitted_models[name] = pipe
    metric_rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })

metrics = pd.DataFrame(metric_rows).sort_values("MAE")
display(metrics.round(4))
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)

In [ ]:
best_name = metrics.iloc[0]["model"]
best_model = fitted_models[best_name]
best_pred = np.clip(best_model.predict(X_test), 0, None)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, best_pred, alpha=0.55)
max_val = max(float(y_test.max()), float(np.max(best_pred)), 1)
axes[0].plot([0, max_val], [0, max_val], linestyle="--")
axes[0].set_xlabel("Actual visits")
axes[0].set_ylabel("Predicted visits")
axes[0].set_title(f"Actual vs predicted — {best_name}")

residuals = y_test.to_numpy() - best_pred
sns.histplot(residuals, bins=30, kde=True, ax=axes[1])
axes[1].set_title("Prediction residuals")
axes[1].set_xlabel("Actual - predicted")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "best_model_diagnostics.png", dpi=160, bbox_inches="tight")
plt.show()

print("Selected model by lowest held-out MAE:", best_name)
print("This is a single hold-out comparison; confirm stability with cross-validation before production use.")

## 8. Model interpretation

Permutation importance estimates how much test-set performance worsens when a feature is shuffled. It is model- and sample-dependent; correlated predictors can share or mask importance. Importance is predictive, not causal.

In [ ]:
# Permutation importance works on the complete pipeline and original feature columns.
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=10, random_state=SEED, n_jobs=-1
)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance)
importance.to_csv(OUTPUT_DIR / "permutation_importance.csv", index=False)

top = importance.head(15).sort_values("importance_mean")
plt.figure(figsize=(9, max(4, 0.35 * len(top))))
plt.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"])
plt.xlabel("Decrease in negative MAE when feature is shuffled")
plt.title(f"Permutation importance — {best_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "permutation_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Robustness and validation checklist

Before treating any result as final, inspect the following:
1. **Target distribution:** compare zero prevalence and maximum visits with the project presentation.
2. **Data definitions:** verify whether age is recorded in years or as a decimal, how health scores are coded, and whether coverage indicators are mutually exclusive.
3. **Missingness and duplicates:** determine whether missing values or duplicates reflect data collection rather than errors.
4. **Generalization:** use repeated cross-validation or a temporal/external test set if available.
5. **Count-specific models:** compare Poisson or Negative Binomial models and inspect overdispersion; consider a hurdle/two-part model if zero visits dominate.
6. **Fairness and utility:** assess error across relevant demographic groups before any operational use.
7. **Clinical governance:** do not use predictions to deny care, determine eligibility, or replace professional judgement.

In [ ]:
# Optional diagnostic: variance-to-mean ratio of the target.
target_mean = y.mean()
target_var = y.var()
print(f"Target mean: {target_mean:.4f}")
print(f"Target variance: {target_var:.4f}")
print(f"Variance / mean: {target_var / target_mean:.4f}" if target_mean > 0 else "Mean is zero.")
print("A ratio substantially above 1 can indicate overdispersion relative to a simple Poisson assumption; assess formally.")

## 10. Save the selected model and run metadata

The saved pipeline includes preprocessing, so new data must use the same original feature columns. Treat serialized models as trusted local artifacts only.

In [ ]:
model_path = MODEL_DIR / "doctor_visits_model.joblib"
joblib.dump(best_model, model_path)

metadata = {
    "target": TARGET,
    "selected_model": best_name,
    "features": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "random_seed": SEED,
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "data_path": str(DATA_PATH),
    "metrics": metrics.to_dict(orient="records"),
}
metadata_path = MODEL_DIR / "model_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved model:", model_path.resolve())
print("Saved metadata:", metadata_path.resolve())

## 11. Reusable inference example

Replace the example row with a real, de-identified record using the same feature columns as training. The pipeline will handle missing values and categorical encoding.

In [ ]:
# Example template only — populate with actual feature values before use.
example_record = X_test.iloc[[0]].copy()
example_prediction = max(0.0, float(best_model.predict(example_record)[0]))
print("Example predicted visit count:", round(example_prediction, 3))
display(example_record)

## 12. Export analysis tables and final reporting notes

The outputs folder contains the principal tables and figures generated by this notebook. When writing the project report, cite the actual values produced by your run, state the dataset source and version, and distinguish descriptive findings from predictive performance.

In [ ]:
# Export a compact profile and cleaned analysis dataset.
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_n": df.nunique(dropna=True)
})
profile.to_csv(OUTPUT_DIR / "data_quality_profile.csv")
df.to_csv(OUTPUT_DIR / "cleaned_doctor_visits.csv", index=False)

summary = {
    "n_rows_after_cleaning": int(len(df)),
    "n_columns": int(df.shape[1]),
    "zero_visit_share": round(float((df[TARGET] == 0).mean()), 4),
    "mean_visits": round(float(df[TARGET].mean()), 4),
    "median_visits": float(df[TARGET].median()),
    "max_visits": float(df[TARGET].max()),
    "best_model_by_mae": best_name,
    "best_model_metrics": metrics.loc[metrics["model"] == best_name].iloc[0].to_dict(),
}
(OUTPUT_DIR / "executive_summary.json").write_text(json.dumps(summary, indent=2, default=float), encoding="utf-8")
display(pd.Series(summary, name="value").to_frame())
print(f"Analysis outputs written to: {OUTPUT_DIR.resolve()}")

---

## Final interpretation template

Complete this section after running all cells:

- **Dataset:** [source, version, date accessed]
- **Rows retained:** [value] after documented cleaning
- **Zero-visit share:** [value]
- **Strongest descriptive associations:** [features and estimates]
- **Best held-out model:** [model] with MAE [value] and RMSE [value]
- **Limitations:** observational data, possible selection bias, target imbalance, potential measurement error, and no causal identification
- **Recommended next step:** validate results with a count-regression model and cross-validation, then review findings with a domain expert.

**Reproducibility note:** Results will vary if the dataset version, preprocessing decisions, package versions, or train/test split changes.